# Advanced Exception Handling and Custom Exceptions in Python

This notebook explores advanced techniques for handling errors and creating custom exceptions in Python. Exception handling is a critical aspect of writing robust, production-ready code.

## Topics Covered:
1. [Understanding Exception Hierarchies](#1.-Understanding-Exception-Hierarchies)
2. [Advanced Try-Except-Else-Finally Patterns](#2.-Advanced-Try-Except-Else-Finally-Patterns)
3. [Chaining Exceptions with raise from](#3.-Chaining-Exceptions-with-raise-from)
4. [Creating Custom Exception Classes](#4.-Creating-Custom-Exception-Classes)
5. [Handling Multiple Exceptions](#5.-Handling-Multiple-Exceptions)
6. [Context Managers for Exception Handling](#6.-Context-Managers-for-Exception-Handling)
7. [Best Practices for Exception Handling](#7.-Best-Practices-for-Exception-Handling)
8. [Real-world Exception Handling Examples](#8.-Real-world-Exception-Handling-Examples)

## 1. Understanding Exception Hierarchies

Python's exceptions are organized in a hierarchy, with `BaseException` as the root. Understanding this hierarchy is essential for effective error handling.

In [ ]:
# Let's visualize the exception hierarchy
import inspect
import pprint

def get_exception_hierarchy(base_exception=BaseException, level=0):
    """Recursively build the exception hierarchy."""
    result = {}
    for subclass in base_exception.__subclasses__():
        result[subclass.__name__] = get_exception_hierarchy(subclass, level + 1)
    return result

# Display the hierarchy
hierarchy = get_exception_hierarchy()
pprint.pprint(hierarchy, width=80, compact=True, depth=3)

### Core Exception Types

Let's explore some of the most common exception types and their uses:

In [ ]:
# Common exception examples
try:
    # TypeError
    result = "42" + 42
except TypeError as e:
    print(f"TypeError: {e}")
    
try:
    # ValueError
    int("hello")
except ValueError as e:
    print(f"ValueError: {e}")
    
try:
    # KeyError
    d = {}
    d["missing_key"]
except KeyError as e:
    print(f"KeyError: {e}")
    
try:
    # IndexError
    lst = [1, 2, 3]
    lst[10]
except IndexError as e:
    print(f"IndexError: {e}")

### Exception Inheritance

Understanding inheritance in exceptions helps when writing exception handlers. Let's see how catching a parent exception also catches its children:

In [ ]:
# Demonstration of exception inheritance
def demonstrate_exception_inheritance():
    exceptions_to_try = [
        ValueError("A value error"),
        TypeError("A type error"),
        KeyError("A key error"),
        Exception("A generic exception")
    ]
    
    for exception in exceptions_to_try:
        try:
            print(f"\nRaising {exception.__class__.__name__}")
            raise exception
        except Exception as e:
            print(f"Caught by 'except Exception': {e}")
            
demonstrate_exception_inheritance()

## 2. Advanced Try-Except-Else-Finally Patterns

Python's try statement has four clauses:
- `try`: The code that might raise an exception
- `except`: Code that executes if a specific exception is raised
- `else`: Code that executes if no exceptions are raised
- `finally`: Code that always executes, regardless of whether an exception was raised

In [ ]:
# Basic try-except-else-finally example
def divide(x, y):
    try:
        result = x / y
    except ZeroDivisionError:
        print("Cannot divide by zero!")
        return None
    else:
        print("Division succeeded")
        return result
    finally:
        print("This always executes")

print("Case 1: Valid division")
result1 = divide(10, 2)
print(f"Result: {result1}\n")

print("Case 2: Division by zero")
result2 = divide(10, 0)
print(f"Result: {result2}")

### EAFP vs LBYL Approaches

Python encourages the "Easier to Ask for Forgiveness than Permission" (EAFP) approach over "Look Before You Leap" (LBYL):

In [ ]:
# LBYL approach (Look Before You Leap)
def lbyl_approach(dictionary, key):
    if key in dictionary:  # Check first
        return dictionary[key]
    else:
        return "Key not found"

# EAFP approach (Easier to Ask for Forgiveness than Permission)
def eafp_approach(dictionary, key):
    try:
        return dictionary[key]  # Try first
    except KeyError:
        return "Key not found"
    
# Example usage
sample_dict = {"a": 1, "b": 2, "c": 3}

print("LBYL approach:")
print(f"Existing key: {lbyl_approach(sample_dict, 'a')}")
print(f"Missing key: {lbyl_approach(sample_dict, 'z')}")

print("\nEAFP approach:")
print(f"Existing key: {eafp_approach(sample_dict, 'a')}")
print(f"Missing key: {eafp_approach(sample_dict, 'z')}")

### Resource Management with try-finally

In [ ]:
# Classic resource management pattern
def process_file_old_way(filename):
    f = None
    try:
        f = open(filename, 'r')
        # Process the file
        content = f.read()
        return content
    except FileNotFoundError:
        print(f"File {filename} not found")
        return None
    finally:
        if f is not None:
            f.close()
            print("File closed successfully")

# Modern approach using context manager (covered later)
def process_file_new_way(filename):
    try:
        with open(filename, 'r') as f:
            content = f.read()
            return content
    except FileNotFoundError:
        print(f"File {filename} not found")
        return None

# Create a test file
with open('test_file.txt', 'w') as f:
    f.write("This is test content")

# Test both approaches
print("Old approach:")
content1 = process_file_old_way('test_file.txt')
print(f"Content: {content1}\n")

print("New approach:")
content2 = process_file_new_way('test_file.txt')
print(f"Content: {content2}")

# Clean up
import os
os.remove('test_file.txt')

## 3. Chaining Exceptions with raise from

Python 3 introduced exception chaining with the `raise from` syntax, which allows you to indicate that one exception is the direct cause of another. This preserves the original traceback, making debugging easier.

In [ ]:
# Without exception chaining
def fetch_data_no_chaining(user_id):
    try:
        # Simulate attempting to fetch data
        if not isinstance(user_id, int):
            raise TypeError("user_id must be an integer")
    except TypeError:
        # Original exception is lost
        raise ValueError("Invalid user ID format")

# With exception chaining using 'raise from'
def fetch_data_with_chaining(user_id):
    try:
        # Simulate attempting to fetch data
        if not isinstance(user_id, int):
            raise TypeError("user_id must be an integer")
    except TypeError as e:
        # Original exception is preserved as the cause
        raise ValueError("Invalid user ID format") from e

# Test without chaining
try:
    fetch_data_no_chaining("not-an-integer")
except ValueError as e:
    print(f"Without chaining: {e}")
    print(f"__cause__ attribute: {e.__cause__}")

# Test with chaining
try:
    fetch_data_with_chaining("not-an-integer")
except ValueError as e:
    print(f"\nWith chaining: {e}")
    print(f"__cause__ attribute: {e.__cause__}")

### Explicitly Suppressing Exception Chaining

You can explicitly suppress the chaining by using `raise ... from None`:

In [ ]:
# Suppressing exception chaining
def fetch_data_suppress_chaining(user_id):
    try:
        if not isinstance(user_id, int):
            raise TypeError("user_id must be an integer")
    except TypeError:
        # Explicitly suppress the cause
        raise ValueError("Invalid user ID format") from None

# Test with suppressed chaining
try:
    fetch_data_suppress_chaining("not-an-integer")
except ValueError as e:
    print(f"With suppressed chaining: {e}")
    print(f"__cause__ attribute: {e.__cause__}")

## 4. Creating Custom Exception Classes

Creating custom exceptions allows you to define specific error types that are relevant to your application domain.

In [ ]:
# Basic custom exception
class MyCustomError(Exception):
    """Base class for exceptions in this module."""
    pass

# More specific custom exception
class ValueTooLargeError(MyCustomError):
    """Raised when the input value is too large."""
    pass

class ValueTooSmallError(MyCustomError):
    """Raised when the input value is too small."""
    pass

# Using custom exceptions
def validate_value(value):
    if value > 100:
        raise ValueTooLargeError("Value cannot be greater than 100")
    elif value < 0:
        raise ValueTooSmallError("Value cannot be negative")
    return "Value is valid"

# Test
test_values = [50, 150, -10]

for val in test_values:
    try:
        result = validate_value(val)
        print(f"Value {val}: {result}")
    except ValueTooLargeError as e:
        print(f"Value {val}: {e}")
    except ValueTooSmallError as e:
        print(f"Value {val}: {e}")

### Advanced Custom Exceptions

Let's create more sophisticated custom exceptions with additional attributes and functionality:

In [ ]:
class DataValidationError(Exception):
    """Exception raised for errors in the data validation process."""
    
    def __init__(self, field, message, value=None):
        self.field = field
        self.message = message
        self.value = value
        super().__init__(f"Field '{field}': {message}")
    
    def as_dict(self):
        return {
            'field': self.field,
            'message': self.message,
            'value': self.value
        }

# Example usage
def validate_user_data(user_data):
    if 'email' not in user_data or not user_data['email']:
        raise DataValidationError('email', 'Email is required')
        
    if 'age' in user_data and user_data['age'] < 18:
        raise DataValidationError('age', 'Must be at least 18 years old', user_data['age'])
    
    return "User data is valid"

# Test cases
test_users = [
    {'name': 'John', 'email': 'john@example.com', 'age': 25},  # Valid
    {'name': 'Alice', 'age': 30},  # Missing email
    {'name': 'Bob', 'email': 'bob@example.com', 'age': 17}  # Too young
]

for i, user in enumerate(test_users):
    try:
        result = validate_user_data(user)
        print(f"User {i+1}: {result}")
    except DataValidationError as e:
        print(f"User {i+1}: Validation failed")
        print(f"Error details: {e.as_dict()}")

## 5. Handling Multiple Exceptions

Python allows you to handle different exceptions with different handlers, or group multiple exceptions in a single handler.

In [ ]:
# Multiple except blocks
def process_data(data):
    try:
        if isinstance(data, list):
            return data[10]  # Might raise IndexError
        elif isinstance(data, dict):
            return data['key']  # Might raise KeyError
        else:
            return int(data)  # Might raise ValueError or TypeError
    except IndexError:
        print("List index out of range")
        return None
    except KeyError:
        print("Dictionary key not found")
        return None
    except (ValueError, TypeError):
        print("Cannot convert data to integer")
        return None

# Test with different data types
test_data = [
    [1, 2, 3],  # List with fewer than 10 elements
    {"not_key": 42},  # Dict without 'key'
    "not a number"  # String that can't be converted to int
]

for data in test_data:
    result = process_data(data)
    print(f"Data: {data}, Result: {result}")

### Using Exception Groups (Python 3.11+)

Python 3.11 introduced exception groups, which allow you to raise and catch multiple exceptions at once. If you're using Python 3.11 or later, you can run the following code:

In [ ]:
# This code works only in Python 3.11+
import sys
if sys.version_info >= (3, 11):
    # Example using ExceptionGroup
    def process_multiple_items(items):
        errors = []
        results = []
        
        for i, item in enumerate(items):
            try:
                # Process each item
                if not isinstance(item, (int, float)):
                    raise TypeError(f"Item {i} is not a number")
                if item < 0:
                    raise ValueError(f"Item {i} is negative")
                
                results.append(item * 2)
            except Exception as e:
                errors.append(e)
        
        if errors:
            raise ExceptionGroup("Errors processing items", errors)
            
        return results
    
    # Test with a list containing both valid and invalid items
    try:
        test_items = [10, -5, "not a number", 7]
        results = process_multiple_items(test_items)
        print(f"Results: {results}")
    except* TypeError as e_group:
        print(f"Type errors: {e_group.exceptions}")
    except* ValueError as e_group:
        print(f"Value errors: {e_group.exceptions}")
else:
    print("ExceptionGroup is only available in Python 3.11+")

## 6. Context Managers for Exception Handling

Context managers (using the `with` statement) are a powerful way to handle resources and ensure proper cleanup, even when exceptions occur.

In [ ]:
# Basic context manager example
import time

class Timer:
    """A context manager for timing code blocks."""
    
    def __enter__(self):
        self.start = time.time()
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end = time.time()
        self.elapsed = self.end - self.start
        
        # Handle exceptions (if any)
        if exc_type is not None:
            print(f"Code block raised an exception: {exc_type.__name__}: {exc_val}")
            print(f"Execution time before exception: {self.elapsed:.6f} seconds")
            # Returning False propagates the exception
            return False
        
        print(f"Code block executed successfully in {self.elapsed:.6f} seconds")
        return True

# Test successful execution
with Timer() as timer:
    # Some code that takes time
    sum(range(1000000))

# Test exception handling
try:
    with Timer() as timer:
        # Some code that will raise an exception
        sum(range(100))
        raise ValueError("Something went wrong")
except ValueError as e:
    print(f"Exception caught outside the context manager: {e}")

### Using contextlib for Simpler Context Managers

The `contextlib` module provides tools for creating context managers more easily:

In [ ]:
import contextlib

# Using @contextmanager decorator
@contextlib.contextmanager
def error_handler(operation_name):
    """A context manager that catches and reports exceptions."""
    try:
        print(f"Starting operation: {operation_name}")
        yield  # This is where the code inside the 'with' block executes
        print(f"Operation {operation_name} completed successfully")
    except Exception as e:
        print(f"Error during {operation_name}: {type(e).__name__}: {e}")
        raise  # Re-raise the exception

# Test successful operation
with error_handler("data processing"):
    # Successful operation
    data = [1, 2, 3]
    result = sum(data)
    print(f"Result: {result}")

# Test failed operation
try:
    with error_handler("division"):
        # Operation that will fail
        result = 10 / 0
        print(f"This won't execute")
except ZeroDivisionError:
    print("Exception was re-raised and caught outside")

### Suppressing Specific Exceptions

In [ ]:
# Using contextlib.suppress
from contextlib import suppress

# Without suppress
print("Without suppress:")
try:
    file_name = "nonexistent_file.txt"
    with open(file_name, 'r') as f:
        content = f.read()
except FileNotFoundError:
    print(f"File {file_name} not found, continuing...")

# With suppress
print("\nWith suppress:")
with suppress(FileNotFoundError):
    file_name = "another_nonexistent_file.txt"
    with open(file_name, 'r') as f:
        content = f.read()
print("Continued execution after attempted file open")

## 7. Best Practices for Exception Handling

Here are some best practices to follow when working with exceptions in Python:

### 1. Be Specific with Exception Types

Rather than catching all exceptions, catch only the specific exceptions you expect and know how to handle.

In [ ]:
# Bad practice: Catching all exceptions
def bad_practice(filename):
    try:
        with open(filename, 'r') as f:
            return f.read()
    except Exception:  # Too broad!
        return "An error occurred"

# Good practice: Catching specific exceptions
def good_practice(filename):
    try:
        with open(filename, 'r') as f:
            return f.read()
    except FileNotFoundError:
        return f"File {filename} not found"
    except PermissionError:
        return f"No permission to read {filename}"

print("Bad practice result:", bad_practice("nonexistent_file.txt"))
print("Good practice result:", good_practice("nonexistent_file.txt"))

### 2. Keep Try Blocks Short

In [ ]:
# Bad practice: Large try block
def bad_large_try_block(data, filename):
    try:
        # Too many operations in one try block
        result = data['key1']['subkey'] * 2
        with open(filename, 'w') as f:
            f.write(str(result))
        return result
    except Exception as e:
        # Hard to tell which operation failed
        print(f"Error: {e}")
        return None

# Good practice: Smaller, focused try blocks
def good_focused_try_blocks(data, filename):
    # First operation with its own try block
    try:
        result = data['key1']['subkey'] * 2
    except KeyError as e:
        print(f"Data access error: {e}")
        return None
    except TypeError as e:
        print(f"Calculation error: {e}")
        return None
    
    # Second operation with its own try block
    try:
        with open(filename, 'w') as f:
            f.write(str(result))
    except IOError as e:
        print(f"File write error: {e}")
        # We can still return the result even if file writing fails
    
    return result

# Test data
test_data = {"key1": {"wrong_key": 10}}

print("Bad practice result:", bad_large_try_block(test_data, "output.txt"))
print("Good practice result:", good_focused_try_blocks(test_data, "output.txt"))

### 3. Don't Use Empty Except Blocks

In [ ]:
# Bad practice: Empty except block
def bad_empty_except(x, y):
    try:
        return x / y
    except:  # Silent failure
        pass  # This is dangerous!

# Good practice: Informative except block
def good_informative_except(x, y):
    try:
        return x / y
    except ZeroDivisionError:
        print("Cannot divide by zero")
        return None

# Test
print("Bad practice result:", bad_empty_except(5, 0))  # Silently returns None
print("Good practice result:", good_informative_except(5, 0))

### 4. Use Logging Instead of Print for Errors

In [ ]:
import logging

# Configure the logging module
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger('exception_demo')

# Use logging instead of print
def divide_with_logging(x, y):
    try:
        result = x / y
        logger.info(f"Successfully divided {x} by {y} to get {result}")
        return result
    except ZeroDivisionError:
        logger.error(f"Failed to divide {x} by zero")
        return None

# Test
divide_with_logging(10, 2)
divide_with_logging(10, 0)

### 5. Avoid Catching BaseException

In [ ]:
# Bad practice: Catching BaseException
def bad_catch_all():
    try:
        # Some code
        x = 1 / 0
    except BaseException:  # This will catch KeyboardInterrupt, SystemExit, etc.
        print("Caught an exception")

# Good practice: Catch specific exceptions or use Exception
def good_catch_specific():
    try:
        # Some code
        x = 1 / 0
    except Exception:  # This won't catch KeyboardInterrupt or SystemExit
        print("Caught a normal exception")

# Don't run this - it makes it hard to interrupt the program
# bad_catch_all()

# This is safer
good_catch_specific()

### 6. Clean up Resources in Finally Blocks or Context Managers

In [ ]:
# Create a sample file for demonstration
with open('resource_demo.txt', 'w') as f:
    f.write('test data')

# Bad practice: Resource might not be released
def bad_resource_handling():
    f = open('resource_demo.txt', 'r')  # Resource acquired
    data = f.read()                     # If this raises an exception...
    f.close()                           # ...this never executes!
    return data

# Good practice: Using finally block
def good_resource_handling_finally():
    f = None
    try:
        f = open('resource_demo.txt', 'r')  # Resource acquired
        data = f.read()
        return data
    finally:
        if f is not None:  # Always executes, even if an exception occurs
            f.close()

# Best practice: Using context manager
def best_resource_handling_context():
    with open('resource_demo.txt', 'r') as f:  # Resource managed automatically
        data = f.read()
    return data  # File is already closed here

# Test
print("Using finally block:", good_resource_handling_finally())
print("Using context manager:", best_resource_handling_context())

# Clean up
import os
os.remove('resource_demo.txt')

## 8. Real-world Exception Handling Examples

Let's look at some practical, real-world examples of exception handling.

### Example 1: API Request Handling

In [ ]:
import requests
import json
import time
from urllib.parse import urlparse

class APIError(Exception):
    """Base exception for API-related errors."""
    pass

class APIConnectionError(APIError):
    """Raised when there's a connection problem with the API."""
    pass

class APIResponseError(APIError):
    """Raised when the API responds with an error."""
    def __init__(self, status_code, message):
        self.status_code = status_code
        self.message = message
        super().__init__(f"API returned {status_code}: {message}")

class APIRateLimitError(APIError):
    """Raised when the API rate limit is exceeded."""
    def __init__(self, reset_time=None):
        self.reset_time = reset_time
        message = "Rate limit exceeded"
        if reset_time:
            message += f", retry after {reset_time} seconds"
        super().__init__(message)

def fetch_api_data(url, max_retries=3, retry_delay=1):
    """Fetch data from an API with retry logic and error handling."""
    retries = 0
    
    # Validate the URL
    try:
        parsed_url = urlparse(url)
        if not parsed_url.scheme or not parsed_url.netloc:
            raise ValueError("Invalid URL format")
    except Exception as e:
        raise ValueError(f"URL validation failed: {e}") from e
    
    while retries < max_retries:
        try:
            response = requests.get(url, timeout=5)
            
            # Handle HTTP errors
            if response.status_code == 429:  # Too Many Requests
                reset_time = int(response.headers.get('Retry-After', 60))
                raise APIRateLimitError(reset_time)
            
            response.raise_for_status()  # Raise exception for 4XX/5XX responses
            
            # Parse JSON response
            try:
                return response.json()
            except json.JSONDecodeError as e:
                raise APIResponseError(200, "Invalid JSON response") from e
                
        except requests.exceptions.Timeout:
            if retries < max_retries - 1:
                retries += 1
                time.sleep(retry_delay * retries)  # Exponential backoff
                continue
            raise APIConnectionError("Request timed out after multiple retries")
            
        except requests.exceptions.ConnectionError:
            if retries < max_retries - 1:
                retries += 1
                time.sleep(retry_delay * retries)
                continue
            raise APIConnectionError("Failed to connect to API")
            
        except requests.exceptions.HTTPError as e:
            status_code = e.response.status_code
            message = str(e)
            raise APIResponseError(status_code, message) from e
            
        except APIRateLimitError as e:
            if retries < max_retries - 1:
                retries += 1
                time.sleep(min(e.reset_time or retry_delay, 10))
                continue
            raise
            
        except Exception as e:
            raise APIError(f"Unexpected error: {str(e)}") from e

# Test the function with a valid API endpoint
try:
    # This is a public API that returns random user data
    data = fetch_api_data("https://jsonplaceholder.typicode.com/users/1")
    print(f"Successfully fetched data for user: {data['name']}")
except APIError as e:
    print(f"API error: {e}")

# Test with an invalid URL
try:
    data = fetch_api_data("invalid-url")
except ValueError as e:
    print(f"URL error: {e}")

# Test with a non-existent endpoint
try:
    data = fetch_api_data("https://jsonplaceholder.typicode.com/nonexistent")
except APIResponseError as e:
    print(f"API response error: {e} (Status code: {e.status_code})")

### Example 2: Database Operations with Error Recovery

In [ ]:
# Simulated database module
class MockDatabase:
    def __init__(self):
        self.data = {}
        self.connection_status = True
    
    def connect(self):
        self.connection_status = True
        return self.connection_status
    
    def disconnect(self):
        self.connection_status = False
        return not self.connection_status
    
    def is_connected(self):
        return self.connection_status
    
    def execute_query(self, query):
        if not self.connection_status:
            raise ConnectionError("Database connection is closed")
            
        if query.startswith("SELECT"):
            # Parse the table name from a simple SELECT query
            parts = query.split()
            if len(parts) >= 4 and parts[1] == "*" and parts[2] == "FROM":
                table_name = parts[3].strip(";")  # Without trailing semicolon
                return self.data.get(table_name, [])
            else:
                raise ValueError(f"Invalid SELECT query: {query}")
                
        elif query.startswith("INSERT"):
            # Very simplified INSERT handling
            if "INTO users VALUES" in query:
                # Extract the user ID and name from the query
                values_part = query.split("VALUES")[1].strip()
                if values_part.startswith("(") and values_part.endswith(")"):
                    values_part = values_part[1:-1]  # Remove parentheses
                    id_str, name_str = values_part.split(",", 1)
                    user_id = int(id_str.strip())
                    user_name = name_str.strip().strip("'").strip('"')
                    
                    if "users" not in self.data:
                        self.data["users"] = []
                    
                    # Check for duplicate ID
                    for user in self.data["users"]:
                        if user["id"] == user_id:
                            raise ValueError(f"Duplicate user ID: {user_id}")
                    
                    self.data["users"].append({"id": user_id, "name": user_name})
                    return True
                else:
                    raise ValueError(f"Invalid VALUES format in query: {query}")
            else:
                raise ValueError(f"Unsupported INSERT query: {query}")
        else:
            raise NotImplementedError(f"Unsupported query type: {query}")

# Custom database exceptions
class DatabaseError(Exception):
    """Base exception for database-related errors."""
    pass

class ConnectionError(DatabaseError):
    """Raised when there's a problem with the database connection."""
    pass

class QueryError(DatabaseError):
    """Raised when there's a problem with a database query."""
    pass

# Database access layer with error handling
class UserRepository:
    def __init__(self, db):
        self.db = db
    
    def ensure_connection(self):
        """Ensure database is connected, reconnecting if needed."""
        max_attempts = 3
        attempts = 0
        
        while attempts < max_attempts:
            if self.db.is_connected():
                return True
                
            try:
                print(f"Attempting to connect to database (attempt {attempts + 1})")
                self.db.connect()
                print("Database connection established")
                return True
            except Exception as e:
                attempts += 1
                if attempts >= max_attempts:
                    raise ConnectionError(f"Failed to connect to database after {max_attempts} attempts") from e
                time.sleep(1)  # Wait before retrying
        
        return False
    
    def get_all_users(self):
        """Fetch all users from the database."""
        try:
            self.ensure_connection()
            return self.db.execute_query("SELECT * FROM users;")
        except ConnectionError as e:
            raise ConnectionError("Failed to retrieve users due to connection error") from e
        except Exception as e:
            raise QueryError(f"Error fetching users: {str(e)}") from e
    
    def add_user(self, user_id, name):
        """Add a new user to the database."""
        if not isinstance(user_id, int) or user_id <= 0:
            raise ValueError("User ID must be a positive integer")
        
        if not name or not isinstance(name, str):
            raise ValueError("User name must be a non-empty string")
        
        query = f"INSERT INTO users VALUES ({user_id}, '{name}')"
        
        try:
            self.ensure_connection()
            self.db.execute_query(query)
            return True
        except ValueError as e:
            if "Duplicate user ID" in str(e):
                # Convert to a more specific exception
                raise QueryError(f"User ID {user_id} already exists") from e
            raise QueryError(f"Invalid user data: {str(e)}") from e
        except ConnectionError as e:
            raise ConnectionError(f"Failed to add user due to connection error") from e
        except Exception as e:
            raise QueryError(f"Error adding user: {str(e)}") from e

# Test the database error handling
db = MockDatabase()
repo = UserRepository(db)

# Test adding users
try:
    repo.add_user(1, "Alice")
    repo.add_user(2, "Bob")
    print("Users added successfully")
except DatabaseError as e:
    print(f"Database error: {e}")

# Test retrieving users
try:
    users = repo.get_all_users()
    print(f"Retrieved {len(users)} users:")
    for user in users:
        print(f"  ID: {user['id']}, Name: {user['name']}")
except DatabaseError as e:
    print(f"Database error: {e}")

# Test adding a duplicate user
try:
    repo.add_user(1, "Duplicate Alice")
    print("User added successfully (shouldn't happen)")
except QueryError as e:
    print(f"Expected error: {e}")

# Test with a disconnected database
db.disconnect()
try:
    users = repo.get_all_users()
    print(f"Retrieved {len(users)} users")
except DatabaseError as e:
    print(f"Expected error: {e}")

## Conclusion

Advanced exception handling and custom exceptions are powerful tools for writing robust, maintainable Python code. By understanding exception hierarchies, implementing proper error handling patterns, and following best practices, you can create code that gracefully handles errors and provides informative feedback when things go wrong.

Key takeaways:

1. Use specific exception types and create custom exceptions for your application domain
2. Understand the relationships between exceptions in the Python hierarchy
3. Use advanced try-except-else-finally patterns appropriately
4. Chain exceptions when appropriate to preserve context
5. Keep try blocks small and focused
6. Use context managers for resource management
7. Follow best practices like avoiding empty except blocks and using proper logging
8. Apply these concepts to real-world scenarios like API calls and database operations

By mastering these techniques, you'll write code that's both more resilient to errors and easier to debug when problems occur.